In [1]:
# Install all Python dependencies
import subprocess, sys

pkgs = [
    "rpy2", "GEOparse", "umap-learn", "statsmodels",
    "matplotlib-venn", "seaborn", "scikit-learn",
]
for p in pkgs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", p, "-q"])
print("✓ Python packages installed")

✓ Python packages installed


In [3]:
import subprocess

# Install R system libraries needed for compilation
subprocess.run("apt-get install -y r-base r-base-dev libcurl4-openssl-dev "
               "libssl-dev libxml2-dev libfontconfig1-dev libharfbuzz-dev "
               "libfribidi-dev libfreetype6-dev libpng-dev libtiff5-dev "
               "libjpeg-dev 2>/dev/null",
               shell=True, capture_output=True)
print("✓ System libraries ready")

# Install all R packages via rpy2
import os
r_home = subprocess.check_output(["R", "RHOME"]).decode().strip()
os.environ["R_HOME"] = r_home
print(f"✓ R_HOME = {r_home}")

import rpy2.robjects as ro

ro.r('''
options(repos=c(CRAN="https://cloud.r-project.org"))

# CRAN packages
cran_pkgs <- c("MatchIt","BiocManager")
for (p in cran_pkgs) {
  if (!requireNamespace(p, quietly=TRUE)) {
    install.packages(p, quiet=TRUE)
    cat("Installed:", p, "\n")
  } else { cat("Already have:", p, "\n") }
}

# Bioconductor packages
bioc_pkgs <- c("DESeq2","edgeR","limma","sva","GEOquery",
               "clusterProfiler","org.Hs.eg.db","BiocParallel")
for (p in bioc_pkgs) {
  if (!requireNamespace(p, quietly=TRUE)) {
    BiocManager::install(p, ask=FALSE, update=FALSE, quiet=TRUE)
    cat("Installed:", p, "\n")
  } else { cat("Already have:", p, "\n") }
}
cat("\n✓ All R packages ready\n")
''')

✓ System libraries ready
✓ R_HOME = /usr/lib/R
Already have: MatchIt 
Already have: BiocManager 
Already have: DESeq2 
Already have: edgeR 
Already have: limma 
Already have: sva 
Already have: GEOquery 
Already have: clusterProfiler 
Already have: org.Hs.eg.db 
Already have: BiocParallel 

✓ All R packages ready


In [4]:
import os, sys, warnings, subprocess
warnings.filterwarnings("ignore")

# Set R_HOME (auto-detect)
r_home = subprocess.check_output(["R", "RHOME"]).decode().strip()
os.environ["R_HOME"] = r_home

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import multipletests
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
import umap
from sklearn.cluster import DBSCAN
import GEOparse
import rpy2.robjects as ro

BASE     = "/content/bioinformatics"
DATA_RAW = f"{BASE}/data/raw"
DATA_PROC= f"{BASE}/data/processed"
FIGURES  = f"{BASE}/results/figures"
TABLES   = f"{BASE}/results/tables"
AD_PROC  = f"{BASE}/data/processed/AD"

for d in [DATA_RAW, DATA_PROC, FIGURES, TABLES, AD_PROC]:
    os.makedirs(d, exist_ok=True)

# ── Thresholds (matching paper) ────────────────────────────────────────────
FDR = 0.05
LFC = 0.40

# ── Helpers ────────────────────────────────────────────────────────────────
def section(t): print(f"\n{'='*60}\n  {t}\n{'='*60}")
def step(m):    print(f"  → {m}")

def is_sig(df, g):
    if g not in df.index: return False
    r   = df.loc[g]
    fdr = r.get("padj", np.nan)
    lfc = r.get("log2FoldChange", np.nan)
    return not (pd.isna(fdr) or pd.isna(lfc)) and fdr < FDR and abs(lfc) >= LFC

def gini(arr):
    arr = np.sort(np.abs(arr)); n = len(arr)
    return 0.0 if arr.sum()==0 else (2*np.sum(np.arange(1,n+1)*arr)-(n+1)*arr.sum())/(n*arr.sum())

def show(path, title="", figsize=None):
    """Display a saved figure inline."""
    from IPython.display import display
    import matplotlib.image as mpimg
    img = mpimg.imread(path)
    h, w = img.shape[:2]
    fw = figsize[0] if figsize else min(14, w/100)
    fh = figsize[1] if figsize else fw * h / w
    fig, ax = plt.subplots(figsize=(fw, fh))
    ax.imshow(img); ax.axis("off")
    if title: ax.set_title(title, fontsize=11, fontweight="bold", pad=8)
    plt.tight_layout(); plt.show()

print(f"✓ Imports ready  |  R_HOME: {r_home}")
print(f"✓ Output dirs created under {BASE}")

✓ Imports ready  |  R_HOME: /usr/lib/R
✓ Output dirs created under /content/bioinformatics


Downloads GSE64810 (HD) count matrix, paper DESeq2 results, and age metadata.
Files go to `/content/bioinformatics/data/raw/`.

In [5]:
section("DATA DOWNLOAD")

# ── GSE64810 normalised count matrix (~5 MB) ─────────────────────────────
counts_gz  = f"{DATA_RAW}/GSE64810_counts.txt.gz"
counts_txt = f"{DATA_RAW}/counts.txt"
if not os.path.exists(counts_txt):
    step("Downloading GSE64810 normalised counts...")
    url = ("https://ftp.ncbi.nlm.nih.gov/geo/series/GSE64nnn/GSE64810/suppl/"
           "GSE64810_mlhd_DESeq2_norm_counts_adjust.txt.gz")
    subprocess.run(["curl", "-L", "--max-time", "300", "-o", counts_gz, url], check=True)
    subprocess.run(["gunzip", "-c", counts_gz], stdout=open(counts_txt, "w"), check=True)
    print(f"    Saved: {counts_txt}")
else:
    print(f"    Already exists: {counts_txt}")

# ── GSE64810 paper DESeq2 results ─────────────────────────────────────────
deseq_gz  = f"{DATA_RAW}/GSE64810_deseq2.txt.gz"
deseq_txt = f"{DATA_RAW}/deseq2_paper.txt"
if not os.path.exists(deseq_txt):
    step("Downloading paper DESeq2 results...")
    url2 = ("https://ftp.ncbi.nlm.nih.gov/geo/series/GSE64nnn/GSE64810/suppl/"
            "GSE64810_mlhd_DESeq2_diffexp_DESeq2_outlier_trimmed_adjust.txt.gz")
    subprocess.run(["curl", "-L", "--max-time", "300", "-o", deseq_gz, url2], check=True)
    subprocess.run(["gunzip", "-c", deseq_gz], stdout=open(deseq_txt, "w"), check=True)
    print(f"    Saved: {deseq_txt}")
else:
    print(f"    Already exists: {deseq_txt}")

# ── GSE64810 metadata via GEOparse ────────────────────────────────────────
step("Fetching GEO metadata for age...")
gse = GEOparse.get_GEO(geo="GSE64810", destdir=DATA_RAW, silent=True)
geo_recs = []
for gsm_name, gsm in gse.gsms.items():
    r = {"geo_id": gsm_name}
    for ch in gsm.metadata.get("characteristics_ch1", []):
        if ":" in ch:
            k, v = ch.split(":", 1)
            r[k.strip().lower().replace(" ", "_")] = v.strip()
    geo_recs.append(r)
geo_meta = pd.DataFrame(geo_recs)

# ── GSE132903 series matrix (~40 MB) ─────────────────────────────────────
ad_gz = f"{DATA_RAW}/GSE132903_series_matrix.txt.gz"
if not os.path.exists(ad_gz):
    step("Downloading GSE132903 (Alzheimer's) series matrix ~40 MB...")
    url3 = ("https://ftp.ncbi.nlm.nih.gov/geo/series/GSE132nnn/GSE132903/"
            "matrix/GSE132903_series_matrix.txt.gz")
    subprocess.run(["curl", "-L", "--max-time", "600", "--retry", "3",
                    "-o", ad_gz, url3], check=True)
    print(f"    Saved: {ad_gz}")
else:
    print(f"    Already exists: {ad_gz}")

step("Loading count matrix...")
counts_raw   = pd.read_csv(counts_txt, sep="\t", index_col=0)
deseq2_paper = pd.read_csv(deseq_txt,  sep="\t", index_col=0)
print(f"    Count matrix:  {counts_raw.shape[0]:,} genes × {counts_raw.shape[1]} samples")
print(f"    Paper DESeq2:  {deseq2_paper.shape[0]:,} genes")
print("✓ All data ready")


  DATA DOWNLOAD
    Already exists: /content/bioinformatics/data/raw/counts.txt
    Already exists: /content/bioinformatics/data/raw/deseq2_paper.txt
  → Fetching GEO metadata for age...
    Already exists: /content/bioinformatics/data/raw/GSE132903_series_matrix.txt.gz
  → Loading count matrix...
    Count matrix:  28,087 genes × 69 samples
    Paper DESeq2:  28,087 genes
✓ All data ready


## Cell 5 — Part 1: Propensity Score Matching (PSM)
Balance HD and control cohorts on **age** using MatchIt (R).

In [6]:
section("PART 1: Propensity Score Matching")

# Build metadata
records = [{"sample_id": col,
            "group":     1 if col.startswith("H_") else 0,
            "condition": "HD" if col.startswith("H_") else "Control"}
           for col in counts_raw.columns]
meta_raw = pd.DataFrame(records)

# Map geo IDs → sample IDs via title field
counts_raw_cols = set(counts_raw.columns)
for gsm_name, gsm in gse.gsms.items():
    title = gsm.metadata.get("title", [""])[0]
    for sid in counts_raw_cols:
        if sid in title:
            idx = geo_meta[geo_meta["geo_id"]==gsm_name].index
            if len(idx): geo_meta.loc[idx[0], "sample_id"] = sid

geo_mapped = geo_meta.dropna(subset=["sample_id"])
meta_raw   = meta_raw.merge(
    geo_mapped[["sample_id","age_of_death"]].rename(columns={"age_of_death":"age"}),
    on="sample_id", how="left")
meta_raw["age"] = pd.to_numeric(meta_raw["age"], errors="coerce")
print(f"  Samples with age: {meta_raw['age'].notna().sum()}/{len(meta_raw)}")
meta_raw.to_csv(f"{DATA_PROC}/metadata_raw.csv", index=False)

# PSM via MatchIt
step("Running PSM (MatchIt nearest-neighbour, seed=42)...")
ro.r(f'''
suppressMessages(library(MatchIt))
meta <- read.csv("{DATA_PROC}/metadata_raw.csv")
meta <- meta[!is.na(meta$age),]
cat("  Before PSM — HD:", sum(meta$group==1), " Ctrl:", sum(meta$group==0), "\n")
set.seed(42)
m       <- matchit(group ~ age, data=meta, method="nearest", ratio=1)
matched <- match.data(m)
cat("  After  PSM — HD:", sum(matched$group==1), " Ctrl:", sum(matched$group==0), "\n")
write.csv(matched, "{DATA_PROC}/metadata_psm.csv", row.names=FALSE)
''')

meta_psm    = pd.read_csv(f"{DATA_PROC}/metadata_psm.csv")
hd_a, ct_a  = meta_psm[meta_psm["group"]==1]["age"].dropna(), meta_psm[meta_psm["group"]==0]["age"].dropna()
_, p        = stats.ttest_ind(hd_a, ct_a)
print(f"  HD:   {hd_a.mean():.1f} ± {hd_a.std():.1f} yrs")
print(f"  Ctrl: {ct_a.mean():.1f} ± {ct_a.std():.1f} yrs")
print(f"  t-test p = {p:.4f}  {'✓ Balanced' if p>0.05 else '⚠ Still significant'}")

matched_ids = meta_psm["sample_id"].tolist()
counts_psm  = counts_raw[[c for c in matched_ids if c in counts_raw.columns]]
meta_psm    = meta_psm[meta_psm["sample_id"].isin(counts_psm.columns)].set_index("sample_id")

# Figure
fig, axes = plt.subplots(1, 2, figsize=(10,4))
for ax, df, title in [
    (axes[0], pd.read_csv(f"{DATA_PROC}/metadata_raw.csv"), "Before PSM"),
    (axes[1], meta_psm.reset_index(),                       "After PSM")]:
    grps = [df[df["group"]==1]["age"].dropna(), df[df["group"]==0]["age"].dropna()]
    bp   = ax.boxplot(grps, labels=["HD","Controls"], patch_artist=True)
    bp["boxes"][0].set_facecolor("#e74c3c"); bp["boxes"][1].set_facecolor("#3498db")
    ax.set_title(f"Age Distribution — {title}", fontweight="bold")
    ax.set_ylabel("Age (years)")
plt.tight_layout()
plt.savefig(f"{FIGURES}/01_psm_age_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"  Final: {counts_psm.shape[1]} matched samples")


  PART 1: Propensity Score Matching
  Samples with age: 69/69
  → Running PSM (MatchIt nearest-neighbour, seed=42)...
  Before PSM — HD: 20  Ctrl: 49 
  After  PSM — HD: 20  Ctrl: 20 
  HD:   58.2 ± 10.4 yrs
  Ctrl: 58.8 ± 10.5 yrs
  t-test p = 0.8802  ✓ Balanced
  Final: 40 matched samples


In [7]:
section("PART 2: QC, Filtering & Outlier Removal")

# Gene QC
step("Filtering ZCG / LCG genes...")
n_total  = len(counts_psm)
zcg_mask = counts_psm.sum(axis=1) == 0
cpm      = counts_psm.div(counts_psm.sum(axis=0), axis=1) * 1e6
lcg_mask = (cpm.mean(axis=1) < 10) & ~zcg_mask
keep     = ~(zcg_mask | lcg_mask)
counts_expr = counts_psm[keep]
print(f"  Total: {n_total:,}  ZCG: {zcg_mask.sum():,}  LCG: {lcg_mask.sum():,}  Kept: {keep.sum():,}")

cats   = ["Total","ZCG\n(removed)","LCG\n(removed)","Expressed\n(kept)"]
vals   = [n_total, zcg_mask.sum(), lcg_mask.sum(), keep.sum()]
colors = ["steelblue","grey","lightcoral","mediumseagreen"]
fig, ax = plt.subplots(figsize=(8,5))
bars = ax.bar(cats, vals, color=colors, edgecolor="black")
for b, v in zip(bars, vals):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+20, f"{v:,}",
            ha="center", va="bottom", fontsize=9)
ax.set_ylabel("Number of Genes")
ax.set_title("Gene-Level QC — Replicating BDASeq® Figure 4", fontweight="bold")
plt.tight_layout()
plt.savefig(f"{FIGURES}/02_gene_qc_waterfall.png", dpi=150, bbox_inches="tight")
plt.show()

# Single-study: skip batch correction
counts_int = counts_expr.round().astype(int).clip(lower=0)
counts_int.to_csv(f"{DATA_PROC}/counts_corrected.csv")
meta_psm.reset_index().to_csv(f"{DATA_PROC}/metadata_aligned.csv", index=False)
print(f"  Batch correction: single study — skipping ComBat-Seq")

# UMAP + DBSCAN
step("UMAP + DBSCAN outlier detection...")
cpm_corr = counts_int.div(counts_int.sum(axis=0), axis=1) * 1e6
log_cpm  = np.log1p(cpm_corr).T
log_cpm  = log_cpm.loc[:, log_cpm.std() > 0]

reducer   = umap.UMAP(n_components=2, random_state=42, n_neighbors=10, min_dist=0.1)
embedding = reducer.fit_transform(StandardScaler().fit_transform(log_cpm))
db_labels = DBSCAN(eps=1.5, min_samples=3).fit_predict(embedding)
n_out, n_in = (db_labels==-1).sum(), (db_labels!=-1).sum()
print(f"  Outliers: {n_out}  |  Inliers: {n_in}")

groups = meta_psm.loc[log_cpm.index, "group"].values
fig, axes = plt.subplots(1, 2, figsize=(13,5))
for ax, colors, title, legend in [
    (axes[0], ["#e74c3c" if l==-1 else "#3498db" for l in db_labels],
     "UMAP — DBSCAN Outliers",
     [mpatches.Patch(color="#3498db", label=f"Inliers (n={n_in})"),
      mpatches.Patch(color="#e74c3c", label=f"Outliers (n={n_out})")]),
    (axes[1], ["#e74c3c" if g==1 else "#3498db" for g in groups],
     "UMAP — HD vs Control",
     [mpatches.Patch(color="#e74c3c", label="HD Cases"),
      mpatches.Patch(color="#3498db", label="Controls")])]:
    ax.scatter(embedding[:,0], embedding[:,1], c=colors, s=50, alpha=0.8,
               edgecolors="k", lw=0.3)
    ax.legend(handles=legend, fontsize=9)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")
plt.tight_layout()
plt.savefig(f"{FIGURES}/02_umap_dbscan.png", dpi=150, bbox_inches="tight")
plt.show()

inliers      = [s for s, l in zip(log_cpm.index, db_labels) if l != -1]
counts_final = counts_int[inliers]
meta_final   = meta_psm.loc[meta_psm.index.isin(inliers)]
counts_final.to_csv(f"{DATA_PROC}/counts_final.csv")
meta_final.reset_index().to_csv(f"{DATA_PROC}/metadata_final.csv", index=False)
print(f"  Final: {counts_final.shape[0]:,} genes × {counts_final.shape[1]} samples")


  PART 2: QC, Filtering & Outlier Removal
  → Filtering ZCG / LCG genes...
  Total: 28,087  ZCG: 0  LCG: 17,302  Kept: 10,785
  Batch correction: single study — skipping ComBat-Seq
  → UMAP + DBSCAN outlier detection...
  Outliers: 0  |  Inliers: 40
  Final: 10,785 genes × 40 samples


In [8]:
section("PART 3: DEA + RMC Consensus")

counts_dea = pd.read_csv(f"{DATA_PROC}/counts_final.csv", index_col=0)
meta_dea   = pd.read_csv(f"{DATA_PROC}/metadata_final.csv").set_index("sample_id")
common     = [c for c in counts_dea.columns if c in meta_dea.index]
counts_dea = counts_dea[common].astype(int).clip(lower=0)
meta_dea   = meta_dea.loc[common]
counts_dea.to_csv(f"{DATA_PROC}/counts_r_input.csv")
meta_dea.reset_index().to_csv(f"{DATA_PROC}/metadata_r_input.csv", index=False)
print(f"  Input: {counts_dea.shape[0]:,} genes × {counts_dea.shape[1]} samples")

step("Running DESeq2...")
ro.r(f'''
suppressMessages(library(DESeq2))
counts <- read.csv("{DATA_PROC}/counts_r_input.csv", row.names=1)
meta   <- read.csv("{DATA_PROC}/metadata_r_input.csv", row.names=1)
counts <- counts[, rownames(meta)]
meta$group <- factor(meta$group, levels=c(0,1))
dds <- DESeqDataSetFromMatrix(countData=round(counts), colData=meta, design=~group)
dds <- DESeq(dds, quiet=TRUE)
res <- as.data.frame(results(dds, contrast=c("group","1","0"), alpha=0.05))
res$gene <- rownames(res)
write.csv(res, "{DATA_PROC}/deseq2_results.csv", row.names=FALSE)
cat("  DESeq2 sig (FDR<0.05):", sum(res$padj<0.05, na.rm=TRUE), "\n")
''')

step("Running edgeR...")
ro.r(f'''
suppressMessages(library(edgeR))
counts <- read.csv("{DATA_PROC}/counts_r_input.csv", row.names=1)
meta   <- read.csv("{DATA_PROC}/metadata_r_input.csv", row.names=1)
counts <- counts[, rownames(meta)]
group  <- factor(meta$group, levels=c(0,1))
y      <- DGEList(counts=round(counts), group=group)
y      <- calcNormFactors(y)
design <- model.matrix(~group)
y      <- estimateDisp(y, design, robust=TRUE)
fit    <- glmQLFit(y, design); qlf <- glmQLFTest(fit, coef=2)
res    <- topTags(qlf, n=Inf)$table; res$gene <- rownames(res)
names(res)[names(res)=="FDR"]   <- "padj"
names(res)[names(res)=="logFC"] <- "log2FoldChange"
write.csv(res, "{DATA_PROC}/edger_results.csv", row.names=FALSE)
cat("  edgeR  sig (FDR<0.05):", sum(res$padj<0.05, na.rm=TRUE), "\n")
''')

step("Running limma-Voom...")
ro.r(f'''
suppressMessages(library(limma)); suppressMessages(library(edgeR))
counts <- read.csv("{DATA_PROC}/counts_r_input.csv", row.names=1)
meta   <- read.csv("{DATA_PROC}/metadata_r_input.csv", row.names=1)
counts <- counts[, rownames(meta)]
group  <- factor(meta$group, levels=c(0,1))
dge    <- DGEList(counts=round(counts), group=group); dge <- calcNormFactors(dge)
design <- model.matrix(~group)
v      <- voom(dge, design, plot=FALSE)
fit    <- lmFit(v, design); fit <- eBayes(fit)
res    <- topTable(fit, coef=2, number=Inf, adjust.method="BH")
res$gene <- rownames(res)
names(res)[names(res)=="adj.P.Val"] <- "padj"
names(res)[names(res)=="logFC"]     <- "log2FoldChange"
write.csv(res, "{DATA_PROC}/limma_results.csv", row.names=FALSE)
cat("  limma  sig (FDR<0.05):", sum(res$padj<0.05, na.rm=TRUE), "\n")
''')

step("Building RMC consensus (majority vote ≥2/3)...")
d2 = pd.read_csv(f"{DATA_PROC}/deseq2_results.csv", index_col="gene")
er = pd.read_csv(f"{DATA_PROC}/edger_results.csv",  index_col="gene")
lv = pd.read_csv(f"{DATA_PROC}/limma_results.csv",  index_col="gene")

recs = []
for g in d2.index.union(er.index).union(lv.index):
    votes = int(is_sig(d2,g)) + int(is_sig(er,g)) + int(is_sig(lv,g))
    lfcs  = [df.loc[g,"log2FoldChange"] for df in [d2,er,lv]
             if g in df.index and not pd.isna(df.loc[g,"log2FoldChange"])]
    fdrs  = [df.loc[g,"padj"] for df in [d2,er,lv]
             if g in df.index and not pd.isna(df.loc[g,"padj"])]
    recs.append({"gene":g,"votes":votes,
                 "mean_log2FC": np.mean(lfcs) if lfcs else np.nan,
                 "best_FDR":    min(fdrs)     if fdrs else np.nan})

rmc  = pd.DataFrame(recs).set_index("gene")
rmc["RMC_DEG"] = rmc["votes"] >= 2
degs = rmc[rmc["RMC_DEG"]]
rmc.to_csv(f"{TABLES}/rmc_deg_results.csv")
print(f"\n  RMC DEGs: {len(degs):,}  UP: {(degs['mean_log2FC']>0).sum()}  DOWN: {(degs['mean_log2FC']<0).sum()}")

d2_paper_sig = set(deseq2_paper[deseq2_paper["padj"] < FDR].index)
d2_ours_sig  = set(d2[d2["padj"] < FDR].index)
overlap      = d2_paper_sig & d2_ours_sig
print(f"  Paper DESeq2 sig: {len(d2_paper_sig):,}  |  Ours: {len(d2_ours_sig):,}  |  Overlap: {len(overlap):,}")


  PART 3: DEA + RMC Consensus
  Input: 10,785 genes × 40 samples
  → Running DESeq2...


  DESeq2 sig (FDR<0.05): 2231 
  → Running edgeR...
  edgeR  sig (FDR<0.05): 1965 
  → Running limma-Voom...
  limma  sig (FDR<0.05): 1947 
  → Building RMC consensus (majority vote ≥2/3)...

  RMC DEGs: 1,172  UP: 728  DOWN: 444
  Paper DESeq2 sig: 5,480  |  Ours: 2,231  |  Overlap: 1,799


In [9]:
section("PART 4: Feature Selection & Validation Plots")

cpm_f    = counts_dea.div(counts_dea.sum(axis=0), axis=1) * 1e6
deg_g    = [g for g in degs.index if g in counts_dea.index]
X        = np.log1p(cpm_f.loc[deg_g]).T
y        = meta_dea["group"].values

step("Gini + Random Forest feature selection...")
gini_s = X.apply(gini, axis=0)
Xs     = StandardScaler().fit_transform(X)
rf     = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(Xs, y)
rf_imp = pd.Series(rf.feature_importances_, index=X.columns)
cv_auc = cross_val_score(rf, Xs, y, cv=5, scoring="roc_auc")
print(f"  RF 5-fold CV AUC: {cv_auc.mean():.3f} ± {cv_auc.std():.3f}  (paper: ~0.97)")

feat = pd.DataFrame({"gini": gini_s, "rf_imp": rf_imp,
                     "mean_log2FC": degs.loc[degs.index.isin(X.columns), "mean_log2FC"]})
for c in ["gini","rf_imp"]:
    feat[c+"_n"] = (feat[c]-feat[c].min())/(feat[c].max()-feat[c].min()+1e-9)
feat["score"] = 0.5*feat["gini_n"] + 0.5*feat["rf_imp_n"]
feat = feat.sort_values("score", ascending=False)
top  = feat.head(max(10, int(len(feat)*0.10)))
top.to_csv(f"{TABLES}/top_druggable_targets.csv")
print(f"  Top druggable targets: {len(top)}")

# ── Volcano plot ────────────────────────────────────────────────────────────
step("Volcano plot...")
plot = d2.dropna(subset=["padj","log2FoldChange"]).copy()
plot["-log10FDR"] = -np.log10(plot["padj"].clip(lower=1e-300))
def cls_v(r):
    if r["padj"]<FDR and r["log2FoldChange"]>=LFC:  return "URG"
    if r["padj"]<FDR and r["log2FoldChange"]<=-LFC: return "DRG"
    return "NS"
plot["class"] = plot.apply(cls_v, axis=1)
pal = {"URG":"#2196F3","DRG":"#F44336","NS":"#BDBDBD"}
fig, ax = plt.subplots(figsize=(9,7))
for c, col in pal.items():
    s = plot[plot["class"]==c]
    ax.scatter(s["log2FoldChange"], s["-log10FDR"], c=col, s=6, alpha=0.6,
               label=f"{c} (n={len(s)})", rasterized=True)
ax.axhline(-np.log10(FDR), ls="--", lw=0.8, c="k", alpha=0.4)
ax.axvline(LFC,  ls="--", lw=0.8, c="k", alpha=0.4)
ax.axvline(-LFC, ls="--", lw=0.8, c="k", alpha=0.4)
for g in ["FTH1","MT2A","NRG1","HMGCS1","SIRT1","NEFH","MAP1B","ROCK1"]:
    if g in plot.index:
        r = plot.loc[g]
        ax.annotate(g, (r["log2FoldChange"], r["-log10FDR"]),
                    fontsize=7, fontweight="bold", xytext=(5,5), textcoords="offset points")
ax.set_xlabel("log2 Fold Change", fontsize=11)
ax.set_ylabel("-log10(FDR)", fontsize=11)
ax.set_title("Volcano Plot — HD vs Control (Replicating BDASeq® Figure 6)",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f"{FIGURES}/04_volcano_plot.png", dpi=200, bbox_inches="tight")
plt.show()

# ── Heatmap ─────────────────────────────────────────────────────────────────
step("Heatmap of top 50 targets...")
top50    = [g for g in top.head(50).index if g in counts_dea.index]
lcpm_top = np.log1p(cpm_f.loc[top50])
sc       = ["#e74c3c" if meta_dea.loc[s,"group"]==1 else "#3498db" for s in lcpm_top.columns]
col_c    = pd.Series(sc, index=lcpm_top.columns).rename("Group")
fig = sns.clustermap(lcpm_top, col_colors=col_c, cmap="RdBu_r", center=0,
                     figsize=(12,14), yticklabels=True, xticklabels=False,
                     z_score=0, cbar_kws={"label":"Z-score log-CPM"})
fig.fig.suptitle("Top 50 Druggable Targets — Replicating BDASeq® Figure 7",
                 y=1.01, fontsize=12, fontweight="bold")
plt.savefig(f"{FIGURES}/04_heatmap_top_targets.png", dpi=150, bbox_inches="tight")
plt.show()

# ── FTH1 validation ──────────────────────────────────────────────────────────
step("FTH1 validation...")
fth1_g = "FTH1"
if fth1_g not in cpm_f.index and "symbol" in deseq2_paper.columns:
    ens = deseq2_paper[deseq2_paper["symbol"]=="FTH1"].index
    fth1_g = ens[0] if len(ens) else None
if fth1_g and fth1_g in cpm_f.index:
    fth1_cpm  = np.log1p(cpm_f.loc[fth1_g])
    hd_e, ct_e = fth1_cpm[meta_dea["group"]==1], fth1_cpm[meta_dea["group"]==0]
    t, p       = stats.ttest_ind(hd_e, ct_e)
    direction  = "DOWN" if hd_e.mean() < ct_e.mean() else "UP"
    fig, ax = plt.subplots(figsize=(6,6))
    bp = ax.boxplot([ct_e.values, hd_e.values], labels=["Controls","HD Cases"],
                    patch_artist=True, medianprops=dict(color="black", linewidth=2))
    bp["boxes"][0].set_facecolor("#3498db"); bp["boxes"][1].set_facecolor("#e74c3c")
    for i, (d, x) in enumerate([(ct_e,1),(hd_e,2)]):
        ax.scatter(np.random.normal(x, 0.07, len(d)), d, alpha=0.5, s=20,
                   color=["#3498db","#e74c3c"][i], zorder=5)
    ax.set_ylabel("log1p(CPM)")
    ax.set_title(f"FTH1 Expression\nt={t:.3f}, p={p:.4f} | {direction}-regulated in HD"
                 f"\nPaper: DOWN in prefrontal cortex ✓", fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{FIGURES}/04_FTH1_expression.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  FTH1: {'✓ CONFIRMED DOWN' if direction=='DOWN' else '✗ Direction mismatch'}")


  PART 4: Feature Selection & Validation Plots
  → Gini + Random Forest feature selection...
  RF 5-fold CV AUC: 0.938 ± 0.056  (paper: ~0.97)
  Top druggable targets: 117
  → Volcano plot...
  → Heatmap of top 50 targets...
  → FTH1 validation...
  FTH1: ✓ CONFIRMED DOWN


In [10]:
section("GO Biological Process Enrichment (Python — memory-safe)")

# ── Install gseapy (lightweight Python alternative to clusterProfiler) ─────
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "gseapy", "-q"])
import gseapy as gp

# ── Map RMC DEG ENSEMBL IDs → gene symbols ────────────────────
rmc_all  = pd.read_csv(f"{TABLES}/rmc_deg_results.csv")
rmc_sig  = rmc_all[(rmc_all["RMC_DEG"] == True) | (rmc_all["RMC_DEG"] == "True")]
rmc_ens  = rmc_sig["gene"].str.replace(r"\..*", "", regex=True).tolist()

paper_deseq_go = pd.read_csv(f"{DATA_RAW}/deseq2_paper.txt", sep="\t", index_col=0)
paper_ens_map  = {idx.split(".")[0]: sym
                  for idx, sym in paper_deseq_go["symbol"].dropna().items()}

gene_symbols = list({paper_ens_map[e] for e in rmc_ens if e in paper_ens_map})
print(f"  Gene symbols from RMC DEGs: {len(gene_symbols)}")

# ── Run Enrichr-based GO:BP enrichment ─────────────────────────
step("Running GO enrichment via Enrichr API...")
try:
    enr = gp.enrichr(
        gene_list=gene_symbols,
        gene_sets=["GO_Biological_Process_2023"],
        organism="human",
        outdir=None,
        no_plot=True
    )
    go_df = enr.results.copy()
    go_df = go_df[go_df["Adjusted P-value"] < 0.05].copy()
    go_df = go_df.rename(columns={"Term": "Term", "Adjusted P-value": "padj"})
    go_df.to_csv(f"{TABLES}/go_enrichment.csv", index=False)
    print(f"  GO terms found: {len(go_df)}")

    if len(go_df) > 0:
        top_go = go_df.nsmallest(20, "padj")[["Term", "padj"]].copy()
        top_go["-log10p"] = -np.log10(top_go["padj"].clip(lower=1e-50))
        top_go["Term"] = top_go["Term"].str.replace(r"\s*\(GO:.*\)", "", regex=True).str[:55]

        fig, ax = plt.subplots(figsize=(10, 8))
        ax.barh(top_go["Term"], top_go["-log10p"],
                color=["#e74c3c" if i % 2 == 0 else "#3498db" for i in range(len(top_go))],
                edgecolor="white", height=0.7)
        ax.set_xlabel("-log10(Adjusted p-value)", fontsize=11)
        ax.set_title("GO Biological Process Enrichment (Replicating BDASeq® Figure 8)",
                     fontsize=12, fontweight="bold")
        ax.invert_yaxis()
        plt.tight_layout()
        plt.savefig(f"{FIGURES}/04_go_enrichment.png", dpi=150, bbox_inches="tight")
        plt.show()
        print(f"  Total significant GO:BP terms: {len(go_df):,}")
    else:
        print("  No significant GO terms at FDR < 0.05")

except Exception as e:
    print(f"  ⚠ Enrichr API error: {e}")
    print("  Tip: If Enrichr is down, re-run this cell in a minute.")



  GO Biological Process Enrichment (Python — memory-safe)
  Gene symbols from RMC DEGs: 1171
  → Running GO enrichment via Enrichr API...
  GO terms found: 68
  Total significant GO:BP terms: 68


In [11]:
section("Accuracy Metrics — Replication vs Paper")

paper_key = ["FTH1","MT1X","MT2A","MT1G","MT1H","MT1M","MMP9",
             "HMGCS1","FASN","DHCR24","CD36","MVD","TM7SF2",
             "NRG1","NRXN3","PRKCG","KCNA1","KCNC3","SYT2",
             "SIRT1","SLC16A12","SLC22A2","SLC38A2","TWIST1",
             "ROCK1","MAP1B","NEFH","SEMA4D","SEMA5B",
             "ADCY1","CAMKK2","P2RX2","VAMP1","SV2C"]

sym2ens      = deseq2_paper["symbol"].reset_index().set_index("symbol")["index"].to_dict()
our_degs_ens = set(rmc[rmc["RMC_DEG"]].index)
our_degs_sym = {sym for sym, ens in sym2ens.items() if ens in our_degs_ens}

found   = [g for g in paper_key if g in our_degs_sym]
missing = [g for g in paper_key if g not in our_degs_sym]
recall  = len(found) / len(paper_key) * 100

print(f'''
+======================================================+
|       REPLICATION ACCURACY vs PAPER                 |
+======================================================+
|  RF CV AUC:              {cv_auc.mean():.3f} +/- {cv_auc.std():.3f}          |
|  Paper key gene recall:  {len(found)}/{len(paper_key)} ({recall:.1f}%)              |
|  DESeq2 overlap w/paper: {len(overlap):,} genes              |
|  RMC DEGs:               {len(degs):,}                      |
+======================================================+
  Found:   {sorted(found)}
  Missing: {sorted(missing)}
''')

pd.Series({"rf_cv_auc": round(cv_auc.mean(),3),
           "recall_pct": round(recall,2),
           "deseq2_overlap": len(overlap),
           "rmc_degs": len(degs)
           }).to_csv(f"{TABLES}/accuracy_metrics.csv", header=["value"])


  Accuracy Metrics — Replication vs Paper

+======================================================+
|       REPLICATION ACCURACY vs PAPER                 |
+======================================================+
|  RF CV AUC:              0.938 +/- 0.056          |
|  Paper key gene recall:  20/34 (58.8%)              |
|  DESeq2 overlap w/paper: 1,799 genes              |
|  RMC DEGs:               1,172                      |
+======================================================+
  Found:   ['ADCY1', 'CAMKK2', 'HMGCS1', 'KCNA1', 'KCNC3', 'MAP1B', 'MT1G', 'MT1M', 'MT1X', 'MT2A', 'MVD', 'NEFH', 'NRXN3', 'ROCK1', 'SEMA5B', 'SIRT1', 'SLC38A2', 'SYT2', 'TM7SF2', 'VAMP1']
  Missing: ['CD36', 'DHCR24', 'FASN', 'FTH1', 'MMP9', 'MT1H', 'NRG1', 'P2RX2', 'PRKCG', 'SEMA4D', 'SLC16A12', 'SLC22A2', 'SV2C', 'TWIST1']



In [12]:
section("PART 5: Novelty — Alzheimer's Disease (GSE132903)")

step("Loading AD series matrix...")
ro.r(f'''
suppressMessages(library(GEOquery))
gse_obj <- getGEO(filename="{DATA_RAW}/GSE132903_series_matrix.txt.gz",
                  GSEMatrix=TRUE, getGPL=FALSE)
expr  <- exprs(gse_obj)
pheno <- pData(gse_obj)
cat("  Expression:", nrow(expr), "probes x", ncol(expr), "samples\n")
write.csv(expr,  "{AD_PROC}/counts_ad_raw.csv")
write.csv(pheno, "{AD_PROC}/meta_ad_raw.csv")
''')

count_ad = pd.read_csv(f"{AD_PROC}/counts_ad_raw.csv", index_col=0)
ad_pheno = pd.read_csv(f"{AD_PROC}/meta_ad_raw.csv",   index_col=0)

# Build metadata
COND = "diagnosis:ch1" if "diagnosis:ch1" in ad_pheno.columns else        next((c for c in ad_pheno.columns if "diagnosis" in c.lower()), ad_pheno.columns[0])
AGE  = "expired_age (years):ch1" if "expired_age (years):ch1" in ad_pheno.columns else        next((c for c in ad_pheno.columns if "age" in c.lower()), None)

def label_ad(x):
    xl = str(x).strip().upper()
    if xl == "AD": return 1
    if xl in ("ND","CONTROL","NORMAL","HEALTHY"): return 0
    xl = xl.lower()
    if any(t in xl for t in ["alzheimer","dementia"]): return 1
    if any(t in xl for t in ["non-demented","neurologically normal"]): return 0
    return -1

meta_ad = pd.DataFrame({
    "sample_id": ad_pheno.index,
    "condition": ad_pheno[COND].astype(str).str.strip(),
    "age": pd.to_numeric(ad_pheno[AGE], errors="coerce") if AGE else np.nan})
meta_ad["group"] = meta_ad["condition"].apply(label_ad)
meta_ad = meta_ad[meta_ad["group"] != -1].reset_index(drop=True)
print(f"  AD: {(meta_ad['group']==1).sum()}  Ctrl: {(meta_ad['group']==0).sum()}")
meta_ad.to_csv(f"{AD_PROC}/metadata_ad_raw.csv", index=False)

# PSM
ro.r(f'''
suppressMessages(library(MatchIt))
meta <- read.csv("{AD_PROC}/metadata_ad_raw.csv")
meta <- meta[!is.na(meta$age),]
cat("  Before PSM — AD:", sum(meta$group==1), "Ctrl:", sum(meta$group==0), "\n")
tryCatch({{
  m <- matchit(group ~ age, data=meta, method="nearest", ratio=1)
  matched <- match.data(m)
  write.csv(matched, "{AD_PROC}/metadata_ad_psm.csv", row.names=FALSE)
  cat("  After PSM:", nrow(matched), "samples\n")
}}, error=function(e){{
  write.csv(meta, "{AD_PROC}/metadata_ad_psm.csv", row.names=FALSE)
  cat("  PSM failed, using all\n")
}})
''')

meta_ad_psm = pd.read_csv(f"{AD_PROC}/metadata_ad_psm.csv")
common_ad   = [c for c in count_ad.columns if c in meta_ad_psm["sample_id"].values]
count_ad_f  = count_ad[common_ad]
meta_ad_f   = meta_ad_psm[meta_ad_psm["sample_id"].isin(common_ad)].set_index("sample_id").loc[common_ad]

# QC + pseudo-count
cpm_ad     = count_ad_f.div(count_ad_f.sum(axis=0), axis=1) * 1e6
keep_ad    = cpm_ad.mean(axis=1) >= 10
count_ad_e = count_ad_f[keep_ad]
count_ad_e = (np.expm1(count_ad_e)*10).round().astype(int).clip(lower=0)              if count_ad_e.values.max() < 100 else count_ad_e.round().astype(int).clip(lower=0)
count_ad_e.to_csv(f"{AD_PROC}/counts_ad_input.csv")
meta_ad_f.reset_index().to_csv(f"{AD_PROC}/metadata_ad_input.csv", index=False)
print(f"  AD final: {count_ad_e.shape[0]:,} probes × {count_ad_e.shape[1]} samples")


  PART 5: Novelty — Alzheimer's Disease (GSE132903)
  → Loading AD series matrix...
  Expression: 42179 probes x 195 samples
  AD: 97  Ctrl: 98
  Before PSM — AD: 92 Ctrl: 92 
  After PSM: 184 samples
  AD final: 42,179 probes × 184 samples


In [13]:
# DEA for AD
step("DEA for AD (3 methods)...")
ro.r(f'''
suppressMessages(library(DESeq2)); suppressMessages(library(edgeR)); suppressMessages(library(limma))
counts <- read.csv("{AD_PROC}/counts_ad_input.csv", row.names=1)
meta   <- read.csv("{AD_PROC}/metadata_ad_input.csv", row.names=1)
common <- intersect(colnames(counts), rownames(meta))
counts <- counts[,common]; meta <- meta[common,,drop=FALSE]
meta$group <- factor(meta$group, levels=c(0,1))
# DESeq2
dds <- DESeqDataSetFromMatrix(countData=round(counts), colData=meta, design=~group)
dds <- DESeq(dds, quiet=TRUE)
res <- as.data.frame(results(dds, contrast=c("group","1","0"), alpha=0.05)); res$gene <- rownames(res)
write.csv(res, "{AD_PROC}/deseq2_ad.csv", row.names=FALSE)
cat("  AD DESeq2:", sum(res$padj<0.05,na.rm=TRUE), "\n")
# edgeR
group <- factor(meta$group,levels=c(0,1)); y <- DGEList(counts=round(counts),group=group)
y <- calcNormFactors(y); design <- model.matrix(~group); y <- estimateDisp(y,design)
fit <- glmQLFit(y,design); qlf <- glmQLFTest(fit,coef=2)
res2 <- topTags(qlf,n=Inf)$table; res2$gene <- rownames(res2)
names(res2)[names(res2)=="FDR"]<-"padj"; names(res2)[names(res2)=="logFC"]<-"log2FoldChange"
write.csv(res2, "{AD_PROC}/edger_ad.csv", row.names=FALSE)
cat("  AD edgeR:", sum(res2$padj<0.05,na.rm=TRUE), "\n")
# limma
dge <- DGEList(counts=round(counts),group=group); dge <- calcNormFactors(dge)
v <- voom(dge,design,plot=FALSE); fit2 <- lmFit(v,design); fit2 <- eBayes(fit2)
res3 <- topTable(fit2,coef=2,number=Inf,adjust.method="BH"); res3$gene <- rownames(res3)
names(res3)[names(res3)=="adj.P.Val"]<-"padj"; names(res3)[names(res3)=="logFC"]<-"log2FoldChange"
write.csv(res3, "{AD_PROC}/limma_ad.csv", row.names=FALSE)
cat("  AD limma:", sum(res3$padj<0.05,na.rm=TRUE), "\n")
''')

# RMC for AD
d2_ad = pd.read_csv(f"{AD_PROC}/deseq2_ad.csv", index_col="gene")
er_ad = pd.read_csv(f"{AD_PROC}/edger_ad.csv",  index_col="gene")
lv_ad = pd.read_csv(f"{AD_PROC}/limma_ad.csv",  index_col="gene")
recs_ad = []
for g in d2_ad.index.union(er_ad.index).union(lv_ad.index):
    votes = sum(is_sig(df,g) for df in [d2_ad,er_ad,lv_ad])
    lfcs  = [df.loc[g,"log2FoldChange"] for df in [d2_ad,er_ad,lv_ad]
             if g in df.index and not pd.isna(df.loc[g,"log2FoldChange"])]
    fdrs  = [df.loc[g,"padj"] for df in [d2_ad,er_ad,lv_ad]
             if g in df.index and not pd.isna(df.loc[g,"padj"])]
    recs_ad.append({"gene":g,"votes":votes,
                    "mean_log2FC":np.mean(lfcs) if lfcs else np.nan,
                    "best_FDR":   min(fdrs)     if fdrs else np.nan})
rmc_ad = pd.DataFrame(recs_ad).set_index("gene")
rmc_ad["RMC_DEG"] = rmc_ad["votes"] >= 2
ad_degs = rmc_ad[rmc_ad["RMC_DEG"]]
rmc_ad.to_csv(f"{TABLES}/rmc_ad_results.csv")
print(f"  AD DEGs: {len(ad_degs):,}  UP:{(ad_degs['mean_log2FC']>0).sum()}  DOWN:{(ad_degs['mean_log2FC']<0).sum()}")

# Feature selection for AD
ad_deg_g = [g for g in ad_degs.index if g in count_ad_e.index]
cpm_adf  = count_ad_e.div(count_ad_e.sum(axis=0), axis=1) * 1e6
X_ad     = np.log1p(cpm_adf.loc[ad_deg_g]).T
y_ad     = meta_ad_f["group"].values
gini_ad  = X_ad.apply(gini, axis=0)
Xs_ad    = StandardScaler().fit_transform(X_ad)
rf_ad    = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_ad.fit(Xs_ad, y_ad)
rf_i     = pd.Series(rf_ad.feature_importances_, index=X_ad.columns)
n_cv     = min(5, min((y_ad==0).sum(), (y_ad==1).sum()))
cv_ad    = cross_val_score(rf_ad, Xs_ad, y_ad, cv=n_cv, scoring="roc_auc")
print(f"  AD RF CV AUC: {cv_ad.mean():.3f} ± {cv_ad.std():.3f}")
feat_ad = pd.DataFrame({"gini":gini_ad,"rf_imp":rf_i,
                        "mean_log2FC":rmc_ad.loc[gini_ad.index,"mean_log2FC"]})
for c in ["gini","rf_imp"]:
    feat_ad[c+"_n"] = (feat_ad[c]-feat_ad[c].min())/(feat_ad[c].max()-feat_ad[c].min()+1e-9)
feat_ad["score"] = 0.5*feat_ad["gini_n"] + 0.5*feat_ad["rf_imp_n"]
feat_ad = feat_ad.sort_values("score", ascending=False)
top_ad  = feat_ad.head(max(10, int(len(feat_ad)*0.10)))
top_ad.to_csv(f"{TABLES}/top_ad_targets.csv")
print(f"  Top AD targets: {len(top_ad)}")

  → DEA for AD (3 methods)...



   function: y = a/x + b, and a local regression fit was automatically substituted.
   specify fitType='local' or 'mean' to avoid this message next time.



  AD DESeq2: 20073 
  AD edgeR: 20272 
  AD limma: 19516 
  AD DEGs: 3,987  UP:2030  DOWN:1957
  AD RF CV AUC: 0.851 ± 0.164
  Top AD targets: 398


## Cell 12 — Novelty: HD vs AD Comparative Target Analysis

In [14]:
section("HD vs AD — Comparative Target Analysis (NOVELTY)")

# Map HD ENSEMBL → symbol
ens2sym          = deseq2_paper["symbol"].dropna().to_dict()
ens2sym_stripped = {k.split(".")[0]: v for k,v in ens2sym.items()}
top_hd_df = pd.read_csv(f"{TABLES}/top_druggable_targets.csv", index_col=0)
hd_syms   = {ens2sym.get(e) or ens2sym_stripped.get(e.split(".")[0])
             for e in top_hd_df.index
             if ens2sym.get(e) or ens2sym_stripped.get(e.split(".")[0])}

# Map AD probe → symbol via GPL10558
gpl_path = f"{DATA_RAW}/GPL10558_annotation.csv"
if not os.path.exists(gpl_path):
    step("Downloading GPL10558 Illumina annotation...")
    ro.r(f'''
    suppressMessages(library(GEOquery))
    gpl <- getGEO("GPL10558", destdir="{DATA_RAW}")
    write.csv(Table(gpl), "{gpl_path}", row.names=FALSE)
    cat("  GPL10558 saved\n")
    ''')
ann     = pd.read_csv(gpl_path)
id_col  = next((c for c in ann.columns if c.upper() in ("ID","PROBE_ID")), ann.columns[0])
sym_col = next((c for c in ann.columns if "SYMBOL" in c.upper()), None) or           next((c for c in ann.columns if "GENE"   in c.upper()), ann.columns[1])
p2sym   = ann.dropna(subset=[id_col,sym_col]).set_index(id_col)[sym_col].to_dict()
ad_syms = {str(p2sym[p]).strip() for p in top_ad.index
           if p in p2sym and str(p2sym[p]).strip() not in ("","nan")}

shared  = hd_syms & ad_syms
hd_only = hd_syms - ad_syms
ad_only = ad_syms - hd_syms

print(f"  HD targets (symbol-mapped):  {len(hd_syms)}")
print(f"  AD targets (symbol-mapped):  {len(ad_syms)}")
print(f"  SHARED:                      {len(shared)}  ← pan-neurodegenerative")
print(f"  HD-specific:                 {len(hd_only)}")
print(f"  AD-specific:                 {len(ad_only)}")
if shared:
    print(f"\n  Shared genes: {sorted(shared)}")

# Venn diagram
try:
    from matplotlib_venn import venn2
except ImportError:
    subprocess.check_call([sys.executable,"-m","pip","install","matplotlib-venn","-q"])
    from matplotlib_venn import venn2

fig, axes = plt.subplots(1, 2, figsize=(14,6))
v = venn2([hd_syms, ad_syms], set_labels=("HD Targets","AD Targets"), ax=axes[0])
for k, col in [("10","#e74c3c"),("01","#3498db"),("11","#9b59b6")]:
    p = v.get_patch_by_id(k)
    if p: p.set_color(col); p.set_alpha(0.7)
axes[0].set_title("HD vs AD — Druggable Target Overlap\n(Key Novelty Finding)", fontweight="bold")

data = {"HD only":len(hd_only), "Shared":len(shared), "AD only":len(ad_only)}
bars = axes[1].bar(data.keys(), data.values(),
                   color=["#e74c3c","#9b59b6","#3498db"], edgecolor="black")
for b, v2 in zip(bars, data.values()):
    axes[1].text(b.get_x()+b.get_width()/2, b.get_height()+0.3, str(v2),
                 ha="center", va="bottom", fontweight="bold", fontsize=13)
axes[1].set_ylabel("Number of Genes")
axes[1].set_title("Target Distribution — HD vs AD", fontweight="bold")
plt.tight_layout()
plt.savefig(f"{FIGURES}/05_hd_vs_ad_venn.png", dpi=150, bbox_inches="tight")
plt.show()


  HD vs AD — Comparative Target Analysis (NOVELTY)
  → Downloading GPL10558 Illumina annotation...
  GPL10558 saved
  HD targets (symbol-mapped):  117
  AD targets (symbol-mapped):  360
  SHARED:                      5  ← pan-neurodegenerative
  HD-specific:                 112
  AD-specific:                 355

  Shared genes: ['CD163', 'EDN1', 'RGS1', 'SLC5A3', 'SST']


In [15]:
section("PIPELINE COMPLETE — FINAL SUMMARY")

print(f"""
╔══════════════════════════════════════════════════════════════╗
║              BDASeq® REPLICATION — ALL RESULTS             ║
╠══════════════════════════════════════════════════════════════╣
║  REPLICATION (HD — GSE64810)                                ║
║  ─────────────────────────────────────────────────────────  ║
║  PSM t-test p:              0.88 ✓ (balanced)              ║
║  Expressed genes after QC:  10,785                         ║
║  UMAP outliers:             0                              ║
║  DESeq2 DEGs:               2,231                          ║
║  edgeR DEGs:                1,965                          ║
║  limma DEGs:                1,947                          ║
║  RMC consensus DEGs:        1,172                          ║
║  RF CV AUC:                 0.938 ✓  (paper: ~0.97)        ║
║  FTH1 direction:            DOWN ✓  (confirmed)            ║
║  Paper key gene recall:     20/34 (58.8%)                  ║
║                                                             ║
║  NOVELTY (AD — GSE132903)                                   ║
║  ─────────────────────────────────────────────────────────  ║
║  AD RMC DEGs:               3,987                          ║
║  AD RF AUC:                 0.852                          ║
║  SHARED HD+AD targets:      5 genes                        ║
║    SST · CD163 · EDN1 · RGS1 · SLC5A3                      ║
╚══════════════════════════════════════════════════════════════╝
""")

# List output files
print("\nOutput files:")
for d in [FIGURES, TABLES]:
    for f in sorted(os.listdir(d)):
        size = os.path.getsize(f"{d}/{f}")
        print(f"  {d}/{f}  ({size//1024} KB)")

# Zip everything for easy download from Colab
import zipfile
zip_path = f"{BASE}/BDASeq_Results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for d in [FIGURES, TABLES]:
        for f in os.listdir(d):
            zf.write(f"{d}/{f}", f"{os.path.basename(d)}/{f}")
print(f"\n✓ Results zipped: {zip_path}")

# Auto-download in Colab
try:
    from google.colab import files
    files.download(zip_path)
    print("✓ Download started — check your browser downloads")
except ImportError:
    print(f"  (Not in Colab — find results at {BASE}/results/)")


  PIPELINE COMPLETE — FINAL SUMMARY

╔══════════════════════════════════════════════════════════════╗
║              BDASeq® REPLICATION — ALL RESULTS             ║
╠══════════════════════════════════════════════════════════════╣
║  REPLICATION (HD — GSE64810)                                ║
║  ─────────────────────────────────────────────────────────  ║
║  PSM t-test p:              0.88 ✓ (balanced)              ║
║  Expressed genes after QC:  10,785                         ║
║  UMAP outliers:             0                              ║
║  DESeq2 DEGs:               2,231                          ║
║  edgeR DEGs:                1,965                          ║
║  limma DEGs:                1,947                          ║
║  RMC consensus DEGs:        1,172                          ║
║  RF CV AUC:                 0.938 ✓  (paper: ~0.97)        ║
║  FTH1 direction:            DOWN ✓  (confirmed)            ║
║  Paper key gene recall:     20/34 (58.8%)                  ║
║          

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Download started — check your browser downloads
